# Import Library


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.svm import SVR
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor


# Data Understanding


In [2]:
df = pd.read_csv(r'DataProduksi.csv')
df

,Tanggal,Minyak Sawit,Karet Kering,Teh
0,Jan-09,1427.10,50.00,8.80
1,Feb-09,1188.00,45.50,7.90
2,Mar-09,1346.70,40.10,8.50
3,Apr-09,1193.50,38.80,9.30
4,May-09,1239.50,47.20,10.30
...,...,...,...,...
115,Aug-18,2075.95,35.32,5.99
116,Sep-18,2093.69,49.37,6.78
117,Oct-18,2075.70,51.21,8.12
118,Nov-18,1931.94,48.77,8.48


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Tanggal       120 non-null    object 
 1   Minyak Sawit  120 non-null    float64
 2   Karet Kering  120 non-null    float64
 3   Teh           120 non-null    float64
dtypes: float64(3), object(1)
memory usage: 3.9+ KB


# Preprocessing


## Change date format


In [4]:
df['Tanggal'] = pd.to_datetime(
    df['Tanggal'], format='%b-%y').dt.strftime('%Y-%m')
df.head()

,Tanggal,Minyak Sawit,Karet Kering,Teh
0,2009-01,1427.1,50.0,8.8
1,2009-02,1188.0,45.5,7.9
2,2009-03,1346.7,40.1,8.5
3,2009-04,1193.5,38.8,9.3
4,2009-05,1239.5,47.2,10.3


## Separating DataFrame Columns into Separate Variables


In [5]:
dates = df['Tanggal'].values
minyak_sawit = df['Minyak Sawit'].values
karet_kering = df['Karet Kering'].values
teh = df['Teh'].values

## Windowed Dataset

In [6]:
def windowed_dataset(data, window_size):
    X = []
    y = []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

## Split Dataset


In [7]:
def split_data(X, y, split_percentage):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=split_percentage, shuffle=False)
    return X_train, X_test, y_train, y_test

## Normalization


In [8]:
scaler = MinMaxScaler()
def normalization(data):
    return scaler.fit_transform(data.reshape(-1, 1))


def to_original_value(data):
    return scaler.inverse_transform(data.reshape(-1, 1))

# Generate Output


## Generate R2 Score


In [9]:
def generate_r2(y_test, y_pred):
    r2 = r2_score(y_test,y_pred)
    print("R2 score: {:.2%}".format(r2))

## Generate Plot

In [10]:
def generate_plot(X_data, y_data, X_name, y_name, attribute_name):
    plt.figure(figsize=(10, 6))
    plt.plot(X_data, label=X_name)
    plt.plot(y_data, label=y_name)
    plt.title(f'{X_name} vs {y_name} {attribute_name}')
    plt.xlabel('Bulan')
    plt.ylabel('Nilai Produksi')
    plt.legend()
    plt.show()
    print('\n')

In [11]:
def generate_plot_lstm(X_data, y_data, X_name, y_name, attribute_name, start_idx=None):
    plt.figure(figsize=(10, 6))
    plt.plot(X_data, label=X_name)
    if start_idx is not None:
        plt.plot(range(start_idx, start_idx + len(y_data)), y_data, label=y_name)
    else:
        plt.plot(y_data, label=y_name)
    plt.title(f'{X_name} vs {y_name} {attribute_name}')
    plt.xlabel('Bulan')
    plt.ylabel('Nilai Produksi')
    plt.legend()
    plt.show()
    print('\n')


## Generate Prediction

### LSTM Predict Future

In [12]:
def predict_future_lstm(model, data, window_size, num_months):
    data_norm = normalization(data)
    last_window = data_norm[-window_size:]
    future_predictions = []
    for i in range(num_months):
        next_prediction = model.predict(last_window.reshape(1, window_size, 1))
        future_predictions.append(next_prediction[0][0])
        last_window = np.append(last_window[1:], next_prediction)[np.newaxis, :]
    return np.array(future_predictions)

### SVR Predict Future

In [13]:
def predict_future_svr(model, data, num_features, num_months):
    data_norm = normalization(data)
    input_seq = data_norm[-num_features:]
    future_preds_norm = []
    for i in range(num_months):
        pred_norm = model.predict(input_seq.reshape(1, -1))
        future_preds_norm.append(pred_norm[0])
        input_seq = np.append(input_seq[1:], pred_norm)
    return to_original_value(np.array(future_preds_norm))

# LSTM


## LSTM Model

In [14]:
def build_model(optimizer='adam', activation='relu', window_size=12, lstm_size=[60,60,60], dense_size=[30,10]):
    model = Sequential()
    for idx, size in enumerate(lstm_size):
        if idx == 0:
            model.add(LSTM(size, return_sequences=True, activation=activation, input_shape=(window_size, 1)))
        elif idx == len(lstm_size) - 1:
            model.add(LSTM(size, activation=activation))
        else:
            model.add(LSTM(size, return_sequences=True, activation=activation))
        # model.add(Dropout(0.2))
    
    for size in dense_size:
        model.add(Dense(size, activation=activation))
    model.add(Dense(1))

    model.compile(loss="mae", optimizer=optimizer, metrics=["mae"])

    return model


In [15]:
def model_lstm(data, attribute_name, split_percentage=0.8, epochs=50, activation='relu', window_size=12, lstm_size=[60,60,60], dense_size=[30,10]):

    X, y = windowed_dataset(normalization(data), window_size=window_size)
    X_train, X_test, y_train, y_test = split_data(X, y, split_percentage)

    model = KerasRegressor(build_fn=build_model, activation=activation, window_size=window_size, lstm_size=lstm_size, dense_size=dense_size, verbose=0)

    param_grid = {'epochs': [50, 100, 150], 
                  'batch_size': [32, 64, 128],
                  'optimizer': ['adam', 'sgd'],
                  'activation': ['relu', 'tanh', 'sigmoid'], 
                  'lstm_size': [[60, 60, 60], [80, 80, 80], [100, 100, 100]], 
                  'dense_size': [[30, 10], [50, 20], [100, 50]]}

    grid = GridSearchCV(model, param_grid=param_grid, cv=3, verbose=0)

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)
    train_pred = best_model.predict(X)
    train_pred_to_plot = np.empty_like(data)
    train_pred_to_plot[:window_size] = np.nan
    train_pred_to_plot[window_size:] = to_original_value(train_pred).flatten()

    generate_r2(y_test, y_pred)

    generate_plot(to_original_value(y_test), to_original_value(y_pred), 'Data Test', 'Data Prediksi', attribute_name)
    generate_plot(data, train_pred_to_plot, 'Data Training','Data Prediksi', attribute_name)

    num_months = [2, 5, 7, 8]
    for n in num_months:
        future_preds = predict_future_lstm(model, minyak_sawit, 1, n)
        print(f"Prediksi {n} bulan ke depan: {future_preds}")

## LSTM Evaluation


### Minyak Sawit


In [16]:
model_lstm(minyak_sawit, 'Minyak Sawit', 0.8, 200, 'relu', 2)


C:\Users\bayui\AppData\Local\Temp\ipykernel_9920\639537839.py:6: DeprecationWarning: KerasRegressor is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasRegressor(build_fn=build_model, activation=activation, window_size=window_size, lstm_size=lstm_size, dense_size=dense_size, verbose=0)


### Karet Kering


In [ ]:
model_lstm(karet_kering, 'Karet Kering', 0.8, 110, 'relu', 12)

### Teh


In [ ]:
model_lstm(teh, 'Teh', 0.8, 100, 'relu', 12)

# SVR (Support Vector Regression)


## SVR Model

In [ ]:
def model_svr(data, attribute_name, split_percentage=0.8, c=1000, gamma='scale', epsilon=0.1):
    X_train, X_test, y_train, y_test = split_data(normalization(data), normalization(data), split_percentage)


    model = SVR(kernel='rbf', C=c, gamma=gamma, epsilon=epsilon)
    model.fit(X_train.reshape(-1,1), y_train.ravel())
    

    y_pred = model.predict(X_test)
    train_pred = model.predict(X_train.reshape(-1,1))

    generate_r2(y_test, y_pred)

    generate_plot(to_original_value(y_test), to_original_value(y_pred), 'Data Test', 'Data Prediksi', attribute_name)
    generate_plot(to_original_value(X_train), to_original_value(train_pred), 'Data Training','Data Prediksi', attribute_name)

    num_months = [2, 5, 7, 8]
    for n in num_months:
        future_preds = predict_future_svr(model, minyak_sawit, 1, n)
        print(f"Prediksi {n} bulan ke depan: {future_preds}")

## SVR Evaluation


### Minyak Sawit


In [ ]:
model_svr(minyak_sawit, 'Minyak Sawit', 0.7, 100, .01, .01)

### Karet Kering


In [ ]:
model_svr(karet_kering, 'Karet Kering', 0.7, 100, .1, .01)

### Teh


In [ ]:
model_svr(teh, 'Teh', 0.8, 100, .1, .01)